### Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [2]:
df = pd.read_excel("data/Data_FE.xlsx", sheet_name="25 Size and BEME portfolios")
df.head()

,Unnamed: 0,Average,Value,Weighted,Returns,--,Monthly,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,Size,Small,Small,Small,Small,Small,2,2.00,2.00,2.00,...,4,4.00,4.00,4.00,4,Big,Big,Big,Big,Big
1,BE/ME,Low,2,3,4,High,Low,2.00,3.00,4.00,...,Low,2.00,3.00,4.00,High,Low,2,3,4,High
2,193601,26.94,15.49,21.82,14.56,33.28,16.43,12.49,9.49,10.27,...,2.13,6.87,8.92,8.35,6.17,2.55,6.36,8.62,14.84,16.29
3,193602,9.46,12.78,6.77,10.02,9,1.7,6.71,5.61,8.92,...,2.59,4.04,6.14,4.80,8.43,1.88,1.18,3.99,3.38,3.75
4,193603,9.46,1.38,5.56,3.01,1.24,-0.37,1.35,3.33,-1.11,...,-0.46,-0.40,3.19,1.52,-4.36,3.58,1.27,-1.82,1.26,-3.56


In [3]:
# Set rows 1 and 2 as MultiIndex header
df.columns = pd.MultiIndex.from_arrays([df.iloc[0], df.iloc[1]])
df = df.iloc[2:].reset_index(drop=True)

# Convert Date column to Year-Month
date_col = df.columns[0]

df[date_col] = pd.to_datetime(df[date_col].astype(int), format="%Y%m")

df = df.set_index(df.columns[0]) 
df = df.apply(pd.to_numeric)

df.head()

0              Small                                  2                       \
1                Low      2      3      4   High    Low      2      3      4   
(Size, BE/ME)                                                                  
1936-01-01     26.94  15.49  21.82  14.56  33.28  16.43  12.49   9.49  10.27   
1936-02-01      9.46  12.78   6.77  10.02   9.00   1.70   6.71   5.61   8.92   
1936-03-01      9.46   1.38   5.56   3.01   1.24  -0.37   1.35   3.33  -1.11   
1936-04-01    -28.60 -29.05 -12.37 -14.03 -21.72 -19.41 -13.35 -15.93 -16.43   
1936-05-01      1.81   8.62   1.89  10.60   5.90   5.21   4.59   7.57   5.84   

0                     ...     4                              Big              \
1               High  ...   Low     2      3      4   High   Low     2     3   
(Size, BE/ME)         ...                                                      
1936-01-01     26.81  ...  2.13  6.87   8.92   8.35   6.17  2.55  6.36  8.62   
1936-02-01      3.87  ...  2.59  4.04   6.14   4.80   8.43  1.88  1.18  3.99   
1936-03-01      1.25  ... -0.46 -0.40   3.19   1.52  -4.36  3.58  1.27 -1.82   
1936-04-01    -17.69  ... -8.31 -9.00 -11.90 -12.09 -12.46 -6.17 -8.26 -7.05   
1936-05-01      6.97  ...  5.12  1.79   4.54   5.98   8.18  4.78  5.20  4.71   

0                            
1                  4   High  
(Size, BE/ME)                
1936-01-01     14.84  16.29  
1936-02-01      3.38   3.75  
1936-03-01      1.26  -3.56  
1936-04-01    -10.57  -8.00  
1936-05-01      6.25   8.39  

[5 rows x 25 columns]

In [4]:
factors = pd.read_excel("data/Data_FE.xlsx", sheet_name="Fama-French factors")

# Convert Date column to Year-Month
date_col = factors.columns[0] 

factors[date_col] = pd.to_datetime(factors[date_col].astype(int), format="%Y%m")
factors = factors.set_index(factors.columns[0]) 
factors = factors.apply(pd.to_numeric)

factors.head()

,Mkt-RF,SMB,HML,RF
Date,,,,
1936-01-01,6.60,6.43,10.09,0.01
1936-02-01,2.56,0.77,3.98,0.01
1936-03-01,0.92,1.10,-2.23,0.02
1936-04-01,-8.07,-6.81,-2.18,0.02
1936-05-01,5.01,0.81,2.69,0.02


In [5]:
target_df = df.copy()

factors = factors.reindex(target_df.index)

target_df = target_df.sub(factors["RF"], axis=0)
target_df.head()

0              Small                                  2                       \
1                Low      2      3      4   High    Low      2      3      4   
(Size, BE/ME)                                                                  
1936-01-01     26.93  15.48  21.81  14.55  33.27  16.42  12.48   9.48  10.26   
1936-02-01      9.45  12.77   6.76  10.01   8.99   1.69   6.70   5.60   8.91   
1936-03-01      9.44   1.36   5.54   2.99   1.22  -0.39   1.33   3.31  -1.13   
1936-04-01    -28.62 -29.07 -12.39 -14.05 -21.74 -19.43 -13.37 -15.95 -16.45   
1936-05-01      1.79   8.60   1.87  10.58   5.88   5.19   4.57   7.55   5.82   

0                     ...     4                              Big              \
1               High  ...   Low     2      3      4   High   Low     2     3   
(Size, BE/ME)         ...                                                      
1936-01-01     26.80  ...  2.12  6.86   8.91   8.34   6.16  2.54  6.35  8.61   
1936-02-01      3.86  ...  2.58  4.03   6.13   4.79   8.42  1.87  1.17  3.98   
1936-03-01      1.23  ... -0.48 -0.42   3.17   1.50  -4.38  3.56  1.25 -1.84   
1936-04-01    -17.71  ... -8.33 -9.02 -11.92 -12.11 -12.48 -6.19 -8.28 -7.07   
1936-05-01      6.95  ...  5.10  1.77   4.52   5.96   8.16  4.76  5.18  4.69   

0                            
1                  4   High  
(Size, BE/ME)                
1936-01-01     14.83  16.28  
1936-02-01      3.37   3.74  
1936-03-01      1.24  -3.58  
1936-04-01    -10.59  -8.02  
1936-05-01      6.23   8.37  

[5 rows x 25 columns]

## Fama-MacBeth Corss-sectional Approach

$$r_{it}=a_i+\beta_{1i}proxy_t^{size}+\beta_{2i}proxy_t^{beme}+\beta_{3i}Market_t+\epsilon_{it}$$

### Linear Regression using Statsmodels package

In [6]:
X = factors[["Mkt-RF", "SMB", "HML"]]
X = sm.add_constant(X)
y = target_df["Small"]["Low"]


model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    Low   R-squared:                       0.826
Model:                            OLS   Adj. R-squared:                  0.826
Method:                 Least Squares   F-statistic:                     1372.
Date:                Thu, 19 Mar 2026   Prob (F-statistic):               0.00
Time:                        02:36:12   Log-Likelihood:                -2391.7
No. Observations:                 870   AIC:                             4791.
Df Residuals:                     866   BIC:                             4811.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.7237      0.131     -5.505      0.0

In [ ]:
X = factors[["Mkt-RF", "SMB", "HML"]]
X = sm.add_constant(X)

r_squa = []

# model.params
# TODO: Get Paramas dataframe too

for j in range(len(target_df.columns)):
    y = target_df.iloc[:, j]
    model = sm.OLS(y, X).fit()
    r_squa.append(model.rsquared)

r_squa_df = pd.DataFrame(r_squa, index=target_df.columns, columns=["R-Squared"])

print(r_squa_df)

            R-Squared
0     1              
Small Low    0.826192
      2      0.890382
      3      0.903950
      4      0.939661
      High   0.927522
2     Low    0.921889
      2      0.937168
      3      0.938942
      4      0.945900
      High   0.954548
3     Low    0.943537
      2      0.913399
      3      0.910198
      4      0.910629
      High   0.921445
4     Low    0.932904
      2      0.909081
      3      0.893939
      4      0.900677
      High   0.887248
Big   Low    0.942769
      2      0.912003
      3      0.862465
      4      0.906117
      High   0.856692


## Macroeconomics Factor Model

In [ ]:
df3 = pd.read_excel("data/Data_FE.xlsx", sheet_name="Macroeconomic factors")

# Convert Date column to Year-Month
df3["Date"] = pd.to_datetime(df3["Date"].astype(int), format="%Y%m")
df3.head()